<table style="width: 100%; border-collapse: collapse; border: none; background: #fffbeb; border-left: 6px solid #f59e0b; border-radius: 8px; padding: 20px; box-shadow: 0 2px 4px rgba(0,0,0,0.05);">
  <tr style="border: none;">
    <td style="vertical-align: middle; border: none; padding: 15px 20px;">
      <h1 style="margin: 0; color: #78350f; font-size: 2em; font-family: system-ui, -apple-system, sans-serif; font-weight: 800; letter-spacing: -0.02em;">
        💡 La Sombra de la Papa: Resumir Muchas Medidas en Pocas 🔦
      </h1>
      <p style="margin: 6px 0 0 0; color: #b45309; font-size: 1.15em; font-weight: 600; font-family: system-ui, -apple-system, sans-serif;">
        Especialización en Ciencia de Datos | Estadística, Análisis y Representación de Datos
      </p>
      <p style="margin: 4px 0 0 0; color: #92400e; font-size: 0.95em; font-family: system-ui, -apple-system, sans-serif;">
        Universidad Santo Tomás — Seccional Tunja
      </p>
    </td>
    <td style="text-align: right; vertical-align: middle; border: none; padding: 15px 20px; width: 30%;">
      <span style="background: #f59e0b; color: #ffffff; padding: 6px 14px; border-radius: 20px; font-size: 0.85em; font-weight: 700; display: inline-block; margin-bottom: 8px;">
        💡 Para Dummies • Módulo 05
      </span><br>
      <span style="color: #78350f; font-size: 0.85em;">Docente: Santiago A. Zúñiga M.</span><br>
      <a href="mailto:gestorvirtualcienciadatos@ustatunja.edu.co" style="color: #b45309; font-size: 0.8em; text-decoration: none; font-weight: 500;">gestorvirtualcienciadatos@ustatunja.edu.co</a>
    </td>
  </tr>
</table>

<div align="center" style="margin-top: 15px; margin-bottom: 15px;">
  <a href="https://colab.research.google.com/github/sazuniga06/Data-Science-Programming---USTA-Tunja-Repository/blob/main/Estadistica%20analisis%20y%20representacion%20de%20datos/05%20-%20Exploracion%20y%20Limpieza%20de%20Datos/Para%20Dummies/02_Reduccion_Dummies.ipynb" target="_parent">
    <img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab" style="vertical-align: middle;"/>
  </a>
</div>

---
## ¿Qué vamos a aprender aquí? 🎈

Este cuaderno es la versión **«para no ingenieros»** del **cuaderno estándar**. Cuando una tabla tiene **muchas columnas** hay dos tareas: **resumirlas en pocas** y **escoger las que de verdad sirven**. Dos analogías:

| Idea | Analogía | Lo que aprenderás |
|---|---|---|
| **Resumir** | La **sombra** de una papa: iluminada desde el mejor ángulo, la sombra conserva casi toda su forma | Con una buena «dirección» se resumen 2 medidas en 1 sin perder casi nada |
| **Escoger** | Buscar entre **200 vendedores** al que «adivina» el clima: alguno acertará por suerte | Si escoges la mejor entre muchas columnas **sin comprobar con datos nuevos**, te engañas |

> 🔗 Si luego quieres ver todo **con rigor y desde cero** (PCA con la SVD, t-SNE, filtro/envoltura/incrustada): **cuaderno estándar**. Antes: **El cuaderno con páginas rotas**.

---
## 1. La Sombra de la Papa: Dos Medidas, Un Resumen 🥔

Medimos **60 plantas**: su **altura** (cm) y el **peso** de la cosecha (g); datos **inventados**. Las plantas grandes tienen más de las dos cosas, así que ambas medidas **cuentan casi lo mismo**. Buscamos **una sola «dirección»** (la mejor para la luz) sobre la que proyectar cada planta; es lo que hace el análisis de componentes principales (**PCA**). Preguntamos: ¿qué parte de la información se conserva con **un solo número** por planta?

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(1)
n = 60
tam = rng.normal(0, 1, n)                                          # "tamaño general" de la planta (no se observa)
altura = 50 + 8 * tam + rng.normal(0, 2, n)                        # cm
peso = 300 + 60 * tam + rng.normal(0, 15, n)                       # gramos
M = np.c_[altura, peso]
Z = (M - M.mean(0)) / M.std(0)                                     # ambas medidas en la misma escala (sin unidades)

U, s, Vt = np.linalg.svd(Z, full_matrices=False)                   # la dirección de la "mejor sombra" es Vt[0]
parte = s[0] ** 2 / (s ** 2).sum()                                 # fracción de la información que conserva UNA sola dirección
resumen_1d = Z @ Vt[0]                                             # un solo número por planta
print(f"Con UN solo número por planta se conserva el {parte:.1%} de la información.")
assert parte > 0.9
assert abs(np.corrcoef(resumen_1d, altura)[0, 1]) > 0.95 and abs(np.corrcoef(resumen_1d, peso)[0, 1]) > 0.95

fig, ax = plt.subplots(figsize=(5.5, 4.5))
ax.scatter(Z[:, 0], Z[:, 1], s=14, color="#2563eb", alpha=0.7)
ax.annotate("", xy=3 * Vt[0], xytext=-3 * Vt[0], arrowprops=dict(arrowstyle="->", color="#dc2626", lw=2))
ax.set_xlabel("altura (estandarizada)"); ax.set_ylabel("peso (estandarizado)"); ax.set_title("La flecha roja es la 'mejor sombra'", fontweight="bold", fontsize=10); ax.set_aspect("equal")
plt.tight_layout(); plt.show()

---
**Lección:** la flecha roja recorre la nube de plantas, y proyectando cada planta sobre ella queda **un solo número que conserva más del 90 %** de la información (y se parece casi totalmente a la altura **y** al peso). Reducir dimensión funciona cuando las columnas **se repiten entre sí**. Ojo: PCA ordena las direcciones por **cuánto varían**, no por **cuánto sirven para predecir**: a veces lo importante está en una dirección que casi no varía (el cuaderno estándar lo muestra).

---

## 2. El Vendedor que «Adivina» el Clima: Escoger Entre Muchas 🎰

Una finca anotó el **rendimiento de 12 cultivos**. Un analista tiene **200 columnas** que **en realidad son puro ruido** (números al azar) y busca **la que más se relacione** con el rendimiento. Con tantas candidatas, alguna parecerá relacionada **solo por suerte**. La prueba de fuego es **repetir con 12 fincas nuevas**: una relación real sobrevive; una casualidad, no.

In [ ]:
rng = np.random.default_rng(3)
rend_a = rng.normal(size=12); cols_a = rng.normal(size=(12, 200))             # 12 fincas, 200 columnas de PURO RUIDO
r_a = np.array([abs(np.corrcoef(cols_a[:, j], rend_a)[0, 1]) for j in range(200)])
mejor = int(r_a.argmax())
print(f"La 'mejor' de 200 columnas de ruido tiene una correlación de {r_a[mejor]:.2f} con el rendimiento. ¡Parece una relación fuerte!")

rend_b = rng.normal(size=12); cols_b = rng.normal(size=(12, 200))             # 12 fincas NUEVAS (y la misma columna 'ganadora')
r_b = abs(np.corrcoef(cols_b[:, mejor], rend_b)[0, 1])
print(f"Con 12 fincas nuevas, esa misma columna tiene una correlación de {r_b:.2f}: la relación se esfumó.")
assert r_a[mejor] > 0.8 and r_b < r_a[mejor] - 0.25

---
**Lección:** entre 200 columnas de ruido, la «ganadora» tiene una correlación **fuerte** (0.85) con apenas 12 fincas; con **fincas nuevas** se **esfuma**. Por eso, antes de quedarte con las «mejores columnas», **compruébalas con datos que no usaste para escogerlas**.

Y una advertencia para el **mapa** que dibuja el método t-SNE (el que agrupa datos «parecidos» en una imagen): es como un **mapa del metro**. Sirve para saber **qué estaciones son vecinas**, pero **no** para medir distancias ni tamaños: dos estaciones que se ven lejos en el dibujo pueden estar cerca en la ciudad, y un «grupo» grande en el dibujo no siempre es un grupo grande en la realidad.

---
##### 🛠️ Práctica 1: ¿Cuánta información conserva una sola medida?

Se miden `largo` y `ancho` de 50 hojas de café (datos **inventados**, ya estandarizados en `W`: cada columna tiene promedio 0 y desviación 1).

1. Calcula `U2, s2, Vt2 = np.linalg.svd(W, full_matrices=False)`.
2. Calcula `parte2 = s2[0] ** 2 / (s2 ** 2).sum()`, la fracción de información que conserva **una sola dirección**.
3. Comprueba con `assert` que `parte2 > 0.8`. Explica con tus palabras por qué una hoja grande «tiene todo grande».

In [ ]:
# Escribe tu código aquí

rng = np.random.default_rng(2)
base = rng.normal(0, 1, 50)
largo = 12 + 2 * base + rng.normal(0, 0.6, 50)
ancho = 5 + 1 * base + rng.normal(0, 0.3, 50)
W = np.c_[largo, ancho]
W = (W - W.mean(0)) / W.std(0)

# U2, s2, Vt2 = ...
# parte2 = ...


<details>
<summary><b>💡 Haz clic aquí para ver la solución guiada...</b></summary>

```python
rng = np.random.default_rng(2)
base = rng.normal(0, 1, 50)
largo = 12 + 2 * base + rng.normal(0, 0.6, 50)
ancho = 5 + 1 * base + rng.normal(0, 0.3, 50)
W = np.c_[largo, ancho]
W = (W - W.mean(0)) / W.std(0)

U2, s2, Vt2 = np.linalg.svd(W, full_matrices=False)
parte2 = s2[0] ** 2 / (s2 ** 2).sum()
print(f"Una sola dirección conserva el {parte2:.1%} de la información.")

assert parte2 > 0.8
print("Una hoja grande suele ser larga Y ancha: las dos medidas cuentan casi lo mismo, así que una sola 'sombra' basta.")
```
</details>

---


---
## Resumen en una frase 🎯

> **Cuando las columnas se repiten entre sí, se pueden resumir en pocas como una buena sombra; pero entre muchas columnas siempre habrá una que «parece» servir por casualidad, así que hay que comprobarla con datos nuevos.**

### Lo que aprendiste hoy:
- ✅ Una **sola dirección** (la «mejor sombra») conservó más del **90 %** de la información de altura y peso, porque ambas medidas se repetían.
- ✅ PCA ordena por **cuánto varía**, no por **cuánto sirve**; lo importante puede estar en una dirección que casi no varía.
- ✅ Entre 200 columnas de ruido, la «mejor» parecía fuerte (0.85) con 12 fincas y **se esfumó** (0.14) con fincas nuevas.
- ✅ Un mapa t-SNE es como un **mapa del metro**: dice quién es vecino, no cuánto mide ni qué tan lejos queda.

> 🎓 **Siguiente paso:** el **cuaderno estándar** lo hace con PCA desde cero, selección de variables con y sin fuga de datos, y las advertencias de t-SNE medidas con datos de verdad conocida.

---
<div align="center">
  <p style="font-size: 0.9em; color: #78350f;">
    © 2026 <b>Universidad Santo Tomás — Seccional Tunja</b><br>
    <i>Especialización en Ciencia de Datos | Estadística, Análisis y Representación de Datos (Edición Didáctica Para Dummies)</i>
  </p>
</div>